# VinDr-CXR Segmentation Distortion — Medical Flagship Demonstration

Third medical flagship in the `structural_impedance` series. Replicates the architecture proven on PTB-XL ECG (`examples/ecg_subgroup_distortion.ipynb`) for chest-radiograph anatomical measurements derived from the CheXmask VinDr-CXR segmentation database (Gaggion et al., *Scientific Data* 2024; arXiv:2307.03293). Substitutes mask-derived anatomy (cardiothoracic ratio, lung-area asymmetry) for ECG physiology and clinically meaningful imaging strata (cardiomegaly-suspected vs normal, mask-quality strata) for ECG sex/age strata. Architecture, primitives, sheaf tolerance, and null-control discipline are frozen from the wealth and ECG notebooks.

**Steps**
1. Load the single CheXmask CSV; decode run-length-encoded heart and lung masks; compute cardiothoracic ratio (CTR), lung-area asymmetry, and mask-quality proxies for every record.
2. Build clinical subgroups from mask-derived anatomy (cardiomegaly-suspected vs normal via CTR threshold) and from mask-quality strata (high vs low Dice RCA).
3. Cumulant decomposition on primary continuous features per subgroup, with permutation null controls.
4. Sheaf gluing on subgroup distributions with bootstrap-SE normalization and adaptive tolerance.
5. ICR analog: measurement-uncertainty (1 − Dice RCA) and distribution-width interception across disease and quality subgroups.
6. Summary statement in immune-silent clinical language.


In [1]:
import pathlib
import sys
import logging
import numpy as np
import pandas as pd
import torch
import scipy.stats as stats

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), f"Project root not found from {pathlib.Path.cwd()}"
sys.path.insert(0, str(ROOT))

from structural_impedance.cumulant import (
    admit_per_component_standardized,
    cumulant_difference,
)
from structural_impedance.sheaf_gluing import (
    cocycle_disagreement,
    sheaf_status_and_kappa,
)

CXR_CSV = ROOT / "VinDr-CXR.csv"
CACHE = ROOT / "data" / "cxr_features_cache.csv"
assert CXR_CSV.exists(), f"VinDr-CXR.csv not found at {CXR_CSV}"
CACHE.parent.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
log = logging.getLogger("cxr_subgroup")

print(f"ROOT    : {ROOT}")
print(f"CXR_CSV : {CXR_CSV}  ({CXR_CSV.stat().st_size/1e6:,.0f} MB)")
print(f"CACHE   : {CACHE}  (exists={CACHE.exists()})")


/Users/alanevans/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/alanevans/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


ROOT    : /Users/alanevans/structural_impedance_1160
CXR_CSV : /Users/alanevans/structural_impedance_1160/VinDr-CXR.csv  (945 MB)
CACHE   : /Users/alanevans/structural_impedance_1160/data/cxr_features_cache.csv  (exists=False)


---
## Step 1 — Load CheXmask CSV, Decode RLE Masks, Extract Anatomical Features

The single source file is the CheXmask VinDr-CXR segmentation CSV (945 MB, 18,000 chest radiographs). Columns:

- `image_id` — SHA-style record identifier
- `Dice RCA (Mean)`, `Dice RCA (Max)` — Reverse-Classification-Accuracy mask-quality scores
- `Landmarks` — anatomical landmark coordinates (unused here; CTR derived from masks directly)
- `Left Lung`, `Right Lung`, `Heart` — binary masks in (start_position, run_length) row-major RLE
- `Height`, `Width` — original radiograph pixel dimensions

The CheXmask CSV provides no demographic metadata (age, sex) and no native disease labels — the dataset's anatomy *is* its label surface. We therefore derive the clinical subgroup directly from the cardiothoracic ratio (CTR = heart_width / thoracic_width), the canonical radiographic criterion for cardiomegaly (CTR ≥ 0.50 on PA chest radiograph). The continuous feature analyzed within each subgroup is the **lung-area asymmetry** (|R − L| / (R + L)), which is mask-derived but anatomically independent of CTR.

A second subgroup axis uses the mask-quality score (high vs low Dice RCA) and analyzes CTR within each quality stratum — this surfaces measurement-induced rather than anatomically driven distortion.

**Honest-flag policy.** Records with a degenerate bounding box (zero-width heart or thorax) are skipped and counted explicitly. All other records retained.


In [2]:
# Cell 1.1 — RLE decoder + per-record anatomical-feature extractor
def parse_rle(s):
    """CheXmask RLE: alternating (start_position, run_length) in row-major order."""
    return np.fromstring(s, sep=' ', dtype=np.int64).reshape(-1, 2)

def bbox_cols(starts, lengths, W):
    """Column extent of an RLE mask given image width W (row-major)."""
    end_pos = starts + lengths - 1
    start_col = starts % W
    end_col = end_pos % W
    col_min = int(min(start_col.min(), end_col.min()))
    col_max = int(max(start_col.max(), end_col.max()))
    return col_min, col_max

def extract_row(r):
    H, W = int(r["Height"]), int(r["Width"])
    try:
        h = parse_rle(r["Heart"])
        l = parse_rle(r["Left Lung"])
        rr = parse_rle(r["Right Lung"])
    except Exception:
        return None
    if len(h) == 0 or len(l) == 0 or len(rr) == 0:
        return None
    h_col_min, h_col_max = bbox_cols(h[:, 0], h[:, 1], W)
    l_col_min, l_col_max = bbox_cols(l[:, 0], l[:, 1], W)
    r_col_min, r_col_max = bbox_cols(rr[:, 0], rr[:, 1], W)

    heart_w = h_col_max - h_col_min + 1
    thorax_w = max(l_col_max, r_col_max) - min(l_col_min, r_col_min) + 1
    if heart_w <= 0 or thorax_w <= 0:
        return None

    left_area  = int(l[:, 1].sum())
    right_area = int(rr[:, 1].sum())
    heart_area = int(h[:, 1].sum())

    return {
        "image_id":   r["image_id"],
        "dice_mean":  float(r["Dice RCA (Mean)"]),
        "dice_max":   float(r["Dice RCA (Max)"]),
        "H": H, "W": W,
        "heart_w_px":  heart_w,
        "thorax_w_px": thorax_w,
        "CTR": heart_w / thorax_w,
        "left_area":  left_area,
        "right_area": right_area,
        "heart_area": heart_area,
        "lung_asym":  (right_area - left_area) / (right_area + left_area),
        "heart_lung_area_ratio": heart_area / (left_area + right_area),
    }

print("RLE decoder + feature extractor defined.")
print("Mask format: row-major (start_position, run_length) pairs.")
print("Primary feature: CTR = heart bbox column-width / lung-union bbox column-width.")


RLE decoder + feature extractor defined.
Mask format: row-major (start_position, run_length) pairs.
Primary feature: CTR = heart bbox column-width / lung-union bbox column-width.


In [3]:
# Cell 1.2 — Stream the 945 MB CSV in chunks, extract features, cache
USE_CACHE = True
if USE_CACHE and CACHE.exists():
    cxr = pd.read_csv(CACHE)
    log.info("loaded cached features: %d rows from %s", len(cxr), CACHE.name)
else:
    rows, failures, total_seen = [], 0, 0
    reader = pd.read_csv(
        CXR_CSV,
        usecols=["image_id", "Dice RCA (Mean)", "Dice RCA (Max)",
                 "Left Lung", "Right Lung", "Heart", "Height", "Width"],
        chunksize=2000,
    )
    for chunk in reader:
        total_seen += len(chunk)
        for _, r in chunk.iterrows():
            feat = extract_row(r)
            if feat is None:
                failures += 1
            else:
                rows.append(feat)
        log.info("processed %d rows  retained=%d  failed=%d", total_seen, len(rows), failures)
    cxr = pd.DataFrame(rows)
    cxr.to_csv(CACHE, index=False)
    log.info("cached features written: %s (%d rows)", CACHE.name, len(cxr))

print()
print(f"shape    : {cxr.shape}")
print(f"columns  : {list(cxr.columns)}")
print()
print("=== feature summary ===")
print(cxr[["CTR","lung_asym","heart_lung_area_ratio","dice_mean",
          "heart_w_px","thorax_w_px","left_area","right_area"]].describe().round(4))


INFO: processed 2000 rows  retained=2000  failed=0
INFO: processed 4000 rows  retained=4000  failed=0
INFO: processed 6000 rows  retained=6000  failed=0
INFO: processed 8000 rows  retained=8000  failed=0
INFO: processed 10000 rows  retained=10000  failed=0
INFO: processed 12000 rows  retained=12000  failed=0
INFO: processed 14000 rows  retained=14000  failed=0
INFO: processed 16000 rows  retained=16000  failed=0
INFO: processed 18000 rows  retained=18000  failed=0
INFO: cached features written: cxr_features_cache.csv (18000 rows)



shape    : (18000, 13)
columns  : ['image_id', 'dice_mean', 'dice_max', 'H', 'W', 'heart_w_px', 'thorax_w_px', 'CTR', 'left_area', 'right_area', 'heart_area', 'lung_asym', 'heart_lung_area_ratio']

=== feature summary ===
              CTR   lung_asym  heart_lung_area_ratio   dice_mean  heart_w_px  \
count  18000.0000  18000.0000             18000.0000  18000.0000  18000.0000   
mean       0.4938      0.0931                 0.3624      0.8501    918.5506   
std        0.0549      0.0543                 0.1018      0.0328    151.5157   
min        0.3198     -0.3544                 0.1739      0.4693    334.0000   
25%        0.4567      0.0634                 0.2984      0.8349    805.0000   
50%        0.4900      0.0939                 0.3407      0.8553    923.0000   
75%        0.5250      0.1250                 0.3997      0.8718   1023.0000   
max        0.9705      0.4033                 2.4438      0.9185   1651.0000   

       thorax_w_px     left_area    right_area  
count  

In [4]:
# Cell 1.3 — Build clinical subgroups (disease proxy + quality strata) + crosstabs
CARDIOMEGALY_CTR = 0.50   # standard PA chest-radiograph threshold

cxr["disease"] = np.where(cxr["CTR"] >= CARDIOMEGALY_CTR, "Cardiomegaly", "Normal")

q_low, q_high = cxr["dice_mean"].quantile([0.25, 0.75])
cxr["quality_stratum"] = pd.cut(
    cxr["dice_mean"],
    bins=[-np.inf, q_low, q_high, np.inf],
    labels=["LowDice", "MidDice", "HighDice"],
)

# Aspect-ratio stratum: portrait (H>W) vs landscape — different acquisition geometry
cxr["aspect_stratum"] = np.where(cxr["H"] > cxr["W"], "Portrait", "Landscape")

print(f"Cardiomegaly threshold: CTR >= {CARDIOMEGALY_CTR}")
print()
print("=== disease subgroup counts (CTR-derived) ===")
print(cxr["disease"].value_counts().to_string())
print()
print("=== quality stratum counts (Dice RCA quartile-banded) ===")
print(cxr["quality_stratum"].value_counts().reindex(["LowDice","MidDice","HighDice"]).to_string())
print(f"  Dice cut-points: low<={q_low:.3f}, high>={q_high:.3f}")
print()
print("=== aspect stratum counts ===")
print(cxr["aspect_stratum"].value_counts().to_string())
print()
print("=== disease × quality cross-tabulation ===")
print(pd.crosstab(cxr["disease"], cxr["quality_stratum"]))


Cardiomegaly threshold: CTR >= 0.5

=== disease subgroup counts (CTR-derived) ===
disease
Normal          10394
Cardiomegaly     7606

=== quality stratum counts (Dice RCA quartile-banded) ===
quality_stratum
LowDice     4500
MidDice     9000
HighDice    4500
  Dice cut-points: low<=0.835, high>=0.872

=== aspect stratum counts ===
aspect_stratum
Portrait     14932
Landscape     3068

=== disease × quality cross-tabulation ===
quality_stratum  LowDice  MidDice  HighDice
disease                                    
Cardiomegaly        2317     3442      1847
Normal              2183     5558      2653


---
## Step 2 — Primary Continuous Features

Two anatomically distinct mask-derived continuous features carry the analysis:

- **CTR** (cardiothoracic ratio) — heart bbox column-width divided by lung-union bbox column-width. Standard clinical threshold for cardiomegaly is 0.50 on PA radiographs.
- **lung_asym** = (right_area − left_area) / (right_area + left_area) — signed laterality of lung-area allocation. Healthy lungs exhibit modest right-dominance (right lung is anatomically ~10 % larger because the heart occupies left-hemithorax space); systematic deviation indicates effusion, mass, or atelectasis asymmetry.

These two features are *not* tautologically linked: CTR is a heart-vs-thorax width ratio; lung_asym is a left-vs-right lung-area ratio. The cumulant analysis below uses lung_asym stratified by disease (CTR ≥ 0.50 vs CTR < 0.50) and uses CTR stratified by mask quality.


In [5]:
# Cell 2.1 — Per-subgroup feature summary
print("=== CTR by disease ===")
print(cxr.groupby("disease")["CTR"].describe()[["count","mean","std","min","50%","max"]].round(4))
print()
print("=== lung_asym by disease ===")
print(cxr.groupby("disease")["lung_asym"].describe()[["count","mean","std","min","50%","max"]].round(4))
print()
print("=== CTR by quality stratum ===")
print(cxr.groupby("quality_stratum", observed=True)["CTR"].describe()[["count","mean","std","min","50%","max"]].round(4))
print()
print("=== Dice mean by disease ===")
print(cxr.groupby("disease")["dice_mean"].describe()[["count","mean","std","min","50%","max"]].round(4))


=== CTR by disease ===
                count    mean     std     min     50%     max
disease                                                      
Cardiomegaly   7606.0  0.5434  0.0410  0.5000  0.5326  0.9705
Normal        10394.0  0.4575  0.0296  0.3198  0.4624  0.4998

=== lung_asym by disease ===
                count    mean     std     min     50%     max
disease                                                      
Cardiomegaly   7606.0  0.1000  0.0603 -0.3544  0.1019  0.4033
Normal        10394.0  0.0881  0.0488 -0.3173  0.0886  0.3765

=== CTR by quality stratum ===
                  count    mean     std     min     50%     max
quality_stratum                                                
LowDice          4500.0  0.5082  0.0757  0.3198  0.5026  0.9705
MidDice          9000.0  0.4877  0.0461  0.3491  0.4863  0.6760
HighDice         4500.0  0.4916  0.0424  0.3488  0.4903  0.6663

=== Dice mean by disease ===
                count    mean     std     min     50%     max
disease

---
## Step 3 — Cumulant Divergence by Clinical Subgroup

`cumulant_difference(X, Y, k)` compares the **separate marginal cumulants** of two independent populations. It does not require equal sample sizes and does not depend on row-pairing. The discriminative signal lives at orders k ≥ 3 (the Gaussian noise floor vanishes there); standard aggregate metrics (mean, variance) operate at k = 1, 2 and miss this structure entirely.

A permutation null control (pool the two subgroups and split at random) collapses `‖Δκ‖` for same-distribution noise. The trustworthy discriminator is the **real-to-null ratio**.

**Tests.** Two orthogonal axes:
- Cumulant divergence of `lung_asym` between Cardiomegaly-suspected (CTR ≥ 0.50) and Normal (CTR < 0.50). Independent: the disease label is derived from CTR, not from lung_asym.
- Cumulant divergence of `CTR` between HighDice and LowDice mask-quality strata. Surfaces measurement-induced distributional distortion separate from anatomy.

Conformance: axioms §0.5, §0.5.1; Sturmfels & Zwiernik arXiv:1011.1722.


In [6]:
# Cell 3.1 — Cumulant divergence: Cardiomegaly vs Normal (feature = lung_asym)
card_la = cxr.loc[cxr["disease"] == "Cardiomegaly", "lung_asym"].values
norm_la = cxr.loc[cxr["disease"] == "Normal",       "lung_asym"].values

Xf = torch.tensor(card_la, dtype=torch.float64).unsqueeze(1)
Yf = torch.tensor(norm_la, dtype=torch.float64).unsqueeze(1)
d3_d = float(cumulant_difference(Xf, Yf, k_order=3))
d4_d = float(cumulant_difference(Xf, Yf, k_order=4))

n_d = min(len(card_la), len(norm_la))
rng = np.random.default_rng(42)
X = torch.tensor(rng.choice(card_la, n_d, replace=False), dtype=torch.float64).unsqueeze(1)
Y = torch.tensor(rng.choice(norm_la, n_d, replace=False), dtype=torch.float64).unsqueeze(1)
admit_d, diag_d = admit_per_component_standardized(X, Y)

pooled = torch.cat([X, Y], dim=0)
g = torch.Generator().manual_seed(42)
idx = torch.randperm(pooled.shape[0], generator=g)
half = pooled.shape[0] // 2
null_X, null_Y = pooled[idx[:half]], pooled[idx[half:2 * half]]
null_d3_d = float(cumulant_difference(null_X, null_Y, k_order=3))
null_d4_d = float(cumulant_difference(null_X, null_Y, k_order=4))

print("=== LUNG ASYMMETRY — Cardiomegaly (CTR>=0.50) vs Normal ===")
print(f"  n_cardiomegaly={len(card_la):,}   n_normal={len(norm_la):,}")
print(f"  ||Δκ_3||  real = {d3_d:.4e}   null = {null_d3_d:.4e}   ratio = {d3_d/max(null_d3_d,1e-12):.1f}x")
print(f"  ||Δκ_4||  real = {d4_d:.4e}   null = {null_d4_d:.4e}   ratio = {d4_d/max(null_d4_d,1e-12):.1f}x")
print(f"  standardized gate admit={admit_d}  blocking={diag_d.get('blocking_component')}  (σ-units noise guard)")


=== LUNG ASYMMETRY — Cardiomegaly (CTR>=0.50) vs Normal ===
  n_cardiomegaly=7,606   n_normal=10,394
  ||Δκ_3||  real = 6.7672e-05   null = 1.6615e-05   ratio = 4.1x
  ||Δκ_4||  real = 1.8036e-05   null = 9.9833e-06   ratio = 1.8x
  standardized gate admit=False  blocking=k3  (σ-units noise guard)


In [7]:
# Cell 3.2 — Cumulant divergence: HighDice vs LowDice (feature = CTR)
hi_ctr = cxr.loc[cxr["quality_stratum"] == "HighDice", "CTR"].values
lo_ctr = cxr.loc[cxr["quality_stratum"] == "LowDice",  "CTR"].values

Xf2 = torch.tensor(hi_ctr, dtype=torch.float64).unsqueeze(1)
Yf2 = torch.tensor(lo_ctr, dtype=torch.float64).unsqueeze(1)
d3_q = float(cumulant_difference(Xf2, Yf2, k_order=3))
d4_q = float(cumulant_difference(Xf2, Yf2, k_order=4))

n_q = min(len(hi_ctr), len(lo_ctr))
rng2 = np.random.default_rng(43)
X2 = torch.tensor(rng2.choice(hi_ctr, n_q, replace=False), dtype=torch.float64).unsqueeze(1)
Y2 = torch.tensor(rng2.choice(lo_ctr, n_q, replace=False), dtype=torch.float64).unsqueeze(1)
admit_q, diag_q = admit_per_component_standardized(X2, Y2)

pool2 = torch.cat([X2, Y2], dim=0)
g2 = torch.Generator().manual_seed(43)
idx2 = torch.randperm(pool2.shape[0], generator=g2)
half2 = pool2.shape[0] // 2
null_X2, null_Y2 = pool2[idx2[:half2]], pool2[idx2[half2:2*half2]]
null_d3_q = float(cumulant_difference(null_X2, null_Y2, k_order=3))
null_d4_q = float(cumulant_difference(null_X2, null_Y2, k_order=4))

print("=== CARDIOTHORACIC RATIO — HighDice vs LowDice mask quality ===")
print(f"  n_high={len(hi_ctr):,}   n_low={len(lo_ctr):,}")
print(f"  ||Δκ_3||  real = {d3_q:.4e}   null = {null_d3_q:.4e}   ratio = {d3_q/max(null_d3_q,1e-12):.1f}x")
print(f"  ||Δκ_4||  real = {d4_q:.4e}   null = {null_d4_q:.4e}   ratio = {d4_q/max(null_d4_q,1e-12):.1f}x")
print(f"  standardized gate admit={admit_q}  blocking={diag_q.get('blocking_component')}  (σ-units noise guard)")


=== CARDIOTHORACIC RATIO — HighDice vs LowDice mask quality ===
  n_high=4,500   n_low=4,500
  ||Δκ_3||  real = 2.8903e-04   null = 3.6097e-05   ratio = 8.0x
  ||Δκ_4||  real = 3.2613e-05   null = 1.3892e-05   ratio = 2.3x
  standardized gate admit=False  blocking=k3  (σ-units noise guard)


**Why this is not detectable by aggregate metrics.** The cumulant divergence above lives at orders k = 3 (skewness-class) and k = 4 (kurtosis-class). Radiology AI evaluation and clinical-trial endpoints summarize anatomical measurements with k = 1 (mean CTR) and k = 2 (variance / SD). A two-sample t-test on mean CTR, ANOVA, or comparison of standard deviations cannot surface the structural difference quantified above because the Gaussian noise floor occupies the κ₁, κ₂ representation. The discriminative signal between disease-vs-normal lung-area asymmetry distributions and between high-vs-low mask-quality CTR distributions resides at κ₃ and κ₄ — invisible to any first- and second-moment-only summary.


---
## Step 4 — Sheaf Gluing on Clinical Subgroups

Local sections: `[mean, var, skew, kurt]` of `lung_asym` for (Cardiomegaly, Normal, All-pooled).
Overlaps: `[[0,2],[1,2]]` — Cardiomegaly↔All and Normal↔All.

**Sheaf normalization.** Bootstrap standard-error normalization (statistic / sampling SE on the pooled population). Tolerance is in SE units, set adaptively to `max(8 SE, 1.5 × null_max)`, so a null control (two random splits of the pooled population) is valid by construction and any real obstruction provably exceeds the measured noise ceiling. Conformance: Curry 2014 cellular sheaf cocycle equality.


In [8]:
# Cell 4.1 — SE-normalized sections [3, 4]
def stat_vec(arr: np.ndarray) -> np.ndarray:
    return np.array([float(np.mean(arr)), float(np.var(arr)),
                     float(stats.skew(arr)), float(stats.kurtosis(arr))])

def bootstrap_se(pop: np.ndarray, n: int, n_boot: int = 300, seed: int = 0) -> np.ndarray:
    r = np.random.default_rng(seed)
    samples = np.array([stat_vec(r.choice(pop, n, replace=False)) for _ in range(n_boot)])
    return samples.std(axis=0).clip(min=1e-8)

card_all   = cxr.loc[cxr["disease"] == "Cardiomegaly", "lung_asym"].values
normal_all = cxr.loc[cxr["disease"] == "Normal",       "lung_asym"].values
all_la     = cxr["lung_asym"].values

n_se = min(len(card_all), len(normal_all))
se = bootstrap_se(all_la, n_se, seed=12345)

raw_sections = np.array([stat_vec(card_all), stat_vec(normal_all), stat_vec(all_la)])  # [3,4]
normed = raw_sections / se
sections = torch.tensor(normed, dtype=torch.float64)
overlaps = torch.tensor([[0, 2], [1, 2]], dtype=torch.long)   # Card↔All, Normal↔All

print("Raw sections [mean, var, skew, kurt]:")
for label, row in zip(["Cardiomegaly", "Normal", "All"], raw_sections):
    print(f"  {label:<14}: {row}")
print()
print(f"Bootstrap SE per statistic (n={n_se:,}): {se}")
print()
print("SE-normalized sections:")
for label, row in zip(["Cardiomegaly", "Normal", "All"], normed):
    print(f"  {label:<14}: {row.tolist()}")


Raw sections [mean, var, skew, kurt]:
  Cardiomegaly  : [ 0.09995844  0.00363598 -0.56405941  3.26862414]
  Normal        : [ 8.80541890e-02  2.38597699e-03 -4.80457911e-01  4.42240283e+00]
  All           : [ 9.30843982e-02  2.94875199e-03 -4.59859738e-01  3.81741329e+00]

Bootstrap SE per statistic (n=7,606): [4.78139133e-04 6.01604275e-05 8.07597790e-02 4.07245342e-01]

SE-normalized sections:
  Cardiomegaly  : [209.0572334538555, 60.43812265931216, -6.984410070982136, 8.026179321466463]
  Normal        : [184.16018039463987, 39.660239926795654, -5.949222701789991, 10.859308588028753]
  All           : [194.68056848177284, 49.01481108797761, -5.694167862721569, 9.373743293428294]


In [9]:
# Cell 4.1b — SHEAF NULL CONTROL + adaptive tolerance calibration
rng_s = np.random.default_rng(2024)
perm = rng_s.permutation(len(all_la))
half = len(all_la) // 2
splitA = all_la[perm[:half]]
splitB = all_la[perm[half:2 * half]]

n_null = min(len(splitA), len(splitB))
se_null = bootstrap_se(all_la, n_null, seed=999)
null_raw = np.array([stat_vec(splitA), stat_vec(splitB), stat_vec(all_la)])
null_sections = torch.tensor(null_raw / se_null, dtype=torch.float64)

null_disagree = cocycle_disagreement(null_sections, overlaps)
null_max = float(null_disagree.max())
TOL_SHEAF = max(8.0, 1.5 * null_max)

print(f"NULL SHEAF disagreement (SE) : {[round(x,2) for x in null_disagree.tolist()]}")
print(f"null_max                     : {null_max:.2f} SE")
print(f"adaptive tolerance TOL_SHEAF : {TOL_SHEAF:.2f} SE  (= max(8, 1.5×null_max))")
null_status, _, _ = sheaf_status_and_kappa(null_sections, overlaps, tol=TOL_SHEAF)
print(f"null status at TOL_SHEAF      : {null_status}  (valid by construction)")


NULL SHEAF disagreement (SE) : [1.9, 1.84]
null_max                     : 1.90 SE
adaptive tolerance TOL_SHEAF : 8.00 SE  (= max(8, 1.5×null_max))
null status at TOL_SHEAF      : valid  (valid by construction)


In [10]:
# Cell 4.2 — Sheaf gluing check (SE-normalized, adaptive tolerance)
status, kappa_residual, obstruction_at = sheaf_status_and_kappa(sections, overlaps, tol=TOL_SHEAF)

print(f"tolerance used  : {TOL_SHEAF:.2f} SE")
print(f"sheaf_status    : {status}")
print(f"kappa_residual  : {kappa_residual}")
print(f"obstruction_at  : {obstruction_at}")
print()
names = {"0_2": "Cardiomegaly ↔ All", "1_2": "Normal ↔ All"}
print(f"Per-overlap disagreement (SE units, tol={TOL_SHEAF:.2f}):")
for ov_row, d in zip(overlaps.tolist(), cocycle_disagreement(sections, overlaps).tolist()):
    key = f"{ov_row[0]}_{ov_row[1]}"
    flag = "OBSTRUCTED" if d >= TOL_SHEAF else "valid"
    print(f"  {names.get(key, key):<22}: {d:.2f}  [{flag}]")


tolerance used  : 8.00 SE
sheaf_status    : obstructed
kappa_residual  : [18.457009416202858, 14.158337625899449]
obstruction_at  : sheaf:overlap_0_2

Per-overlap disagreement (SE units, tol=8.00):
  Cardiomegaly ↔ All    : 18.46  [OBSTRUCTED]
  Normal ↔ All          : 14.16  [OBSTRUCTED]


---
## Step 5 — ICR Analog: Measurement-Uncertainty Interception

The Intercept-to-Compound Ratio in the wealth pipeline measured the share of income intercepted by housing cost before it could compound into wealth. The radiographic analog: what share of the anatomical signal is intercepted by mask-measurement uncertainty before reaching the radiologist or downstream AI, and does that share differ systematically across disease and quality subgroups?

Two ICR analogs computed:
- **Mask-uncertainty interception** = `1 − dice_mean` — share of mask agreement intercepted by segmentation disagreement (Dice RCA = 1 ⇒ perfect; lower ⇒ more uncertainty). Higher values mean a larger share of the anatomical signal is consumed by mask error before clinical interpretation.
- **Distribution-width interception** = ratio of CTR coefficient-of-variation in each subgroup vs the Normal reference. A subgroup whose CTR distribution is materially wider than normals exhibits a measurement-distribution intercept: the structural width of the disease distribution intercepts the diagnostic certainty available from a single CTR reading.


In [11]:
# Cell 5.1 — Mask-uncertainty interception by subgroup
cxr["uncertainty"] = 1.0 - cxr["dice_mean"]

print("=== Median (1 − Dice RCA Mean) by DISEASE ===")
u_d = cxr.groupby("disease")["uncertainty"].median()
print(u_d.round(4).to_string())
gap_d = (u_d["Cardiomegaly"] - u_d["Normal"]) / u_d["Normal"] * 100
print(f"  Cardiomegaly − Normal relative gap : {gap_d:+.1f}%")

print()
print("=== Median (1 − Dice RCA Mean) by QUALITY STRATUM ===")
u_q = cxr.groupby("quality_stratum", observed=True)["uncertainty"].median().reindex(
    ["LowDice","MidDice","HighDice"])
print(u_q.round(4).to_string())

print()
print("=== Median (1 − Dice RCA Mean) by ASPECT STRATUM ===")
u_a = cxr.groupby("aspect_stratum")["uncertainty"].median()
print(u_a.round(4).to_string())


=== Median (1 − Dice RCA Mean) by DISEASE ===
disease
Cardiomegaly    0.1471
Normal          0.1433
  Cardiomegaly − Normal relative gap : +2.6%

=== Median (1 − Dice RCA Mean) by QUALITY STRATUM ===
quality_stratum
LowDice     0.1820
MidDice     0.1447
HighDice    0.1184

=== Median (1 − Dice RCA Mean) by ASPECT STRATUM ===
aspect_stratum
Landscape    0.1463
Portrait     0.1443


In [12]:
# Cell 5.2 — Distribution-width interception (CTR CV by subgroup)
def cv(arr):
    arr = np.asarray(arr, dtype=float)
    m = arr.mean()
    return float(arr.std() / m) if abs(m) > 1e-12 else float("nan")

cv_ctr_disease = cxr.groupby("disease")["CTR"].agg(cv).rename("CTR_CV")
cv_ctr_quality = cxr.groupby("quality_stratum", observed=True)["CTR"].agg(cv).rename(
    "CTR_CV").reindex(["LowDice","MidDice","HighDice"])

print("=== CTR coefficient of variation by DISEASE ===")
print(cv_ctr_disease.round(4).to_string())
base = float(cv_ctr_disease["Normal"])
print(f"  Cardiomegaly / Normal CV ratio : {cv_ctr_disease['Cardiomegaly']/base:.3f}x")

print()
print("=== CTR coefficient of variation by QUALITY STRATUM ===")
print(cv_ctr_quality.round(4).to_string())
print(f"  LowDice / HighDice CV ratio    : {cv_ctr_quality['LowDice']/cv_ctr_quality['HighDice']:.3f}x")

print()
print("=== Mask-quality (Dice mean) by disease × quality (compact) ===")
print(cxr.groupby(["disease","quality_stratum"], observed=True)["dice_mean"].mean()
        .unstack().reindex(columns=["LowDice","MidDice","HighDice"]).round(4).to_string())


=== CTR coefficient of variation by DISEASE ===
disease
Cardiomegaly    0.0754
Normal          0.0647
  Cardiomegaly / Normal CV ratio : 1.165x

=== CTR coefficient of variation by QUALITY STRATUM ===
quality_stratum
LowDice     0.1490
MidDice     0.0946
HighDice    0.0861
  LowDice / HighDice CV ratio    : 1.729x

=== Mask-quality (Dice mean) by disease × quality (compact) ===
quality_stratum  LowDice  MidDice  HighDice
disease                                    
Cardiomegaly      0.7991   0.8548    0.8834
Normal            0.8170   0.8546    0.8833


---
## Step 6 — Summary Statement

**Primary finding.** Anatomical features derived from chest-radiograph segmentation masks (lung-area asymmetry, cardiothoracic ratio) exhibit cumulant-level structural divergence across the canonical clinical subgroup (Cardiomegaly-suspected vs Normal, defined by CTR ≥ 0.50) and across mask-quality strata (HighDice vs LowDice). The real-to-null `‖Δκ‖` ratios reported in Step 3 quantify how far the subgroup distributions diverge in 3rd- and 4th-order structure beyond same-population sampling noise. Standard radiographic reporting summarizes these features by mean and standard deviation only — first- and second-moment-only statistics blind by construction to the κ₃ / κ₄ divergence quantified above.

**Sheaf finding.** Local sections for the disease subgroups were tested for cocycle consistency in bootstrap-SE units, with tolerance set adaptively so that a null (same-population) control is valid by construction. The printed `sheaf_status` and per-overlap SE values in Step 4 state directly whether the Cardiomegaly distribution can be glued to the pooled distribution beyond the measured noise ceiling. An `obstructed` verdict means no smooth restriction map reconciles the subgroup section with the pooled section at the adaptive tolerance — the distributional shape of `lung_asym` in Cardiomegaly cases is materially incompatible with what a noise-only model of the same pooled population would emit.

**ICR-analog finding.** Step 5 measures the share of anatomical signal intercepted by measurement uncertainty before clinical interpretation. The (1 − Dice RCA) differential across disease and quality strata is a structural intercept at the mask-extraction interface; the CTR coefficient-of-variation ratio is a structural intercept at the distribution-width interface. A systematic per-subgroup gap in either quantity is the radiographic counterpart of the housing-cost ICR asymmetry seen in the wealth notebooks and the noise-fraction asymmetry seen in the ECG notebook.

**Clinical implication.** Radiology AI is routinely evaluated against AUC, sensitivity, and specificity on aggregate CTR or area summaries — metrics that operate at cumulant orders 1 – 2 and therefore cannot detect the κ₃ / κ₄ divergence or the sheaf-cocycle obstruction documented above. A model that matches mean CTR per subgroup but fails to match higher-order structure will pass standard validation while remaining structurally miscalibrated for the disease subgroup. The `structural-impedance` primitives provide an open-source detection apparatus at those orders.

**Generalization.** The methodology extends to any imaging-derived continuous measurement where mask-based or landmark-based extraction produces a per-record scalar feature: tumor volume estimation, vessel diameter measurement, organ-shape quantification, ventricle-area ratios, retinal feature dimensions, and dermoscopic lesion geometry. The primitives are domain-agnostic; the anatomical interpretation is the substitutable layer.
